# 🖥️ GPUBid: Agentic GPU Marketplace

Buyer agents represent AI/ML jobs. Seller agents represent GPU capacity slots. They negotiate over rounds via structured offer tuples plus free-form reasoning. A central engine clears agreed deals.

This notebook is the canonical artifact: every cell produces a visual or interactive output. Scroll top-to-bottom to follow the story.

## Setup

Run once when you open the notebook. In Colab this also enables custom widgets so `ipywidgets` work inline.

In [ ]:
# Setup. Behaviour by environment:
#   - Colab: deletes any old clone, re-clones from GitHub, AND evicts any cached
#     `gpubid.*` modules from `sys.modules` so re-running this cell after a code
#     update genuinely picks up the new classes (Python doesn\'t auto-reload).
#     Comment out the `rm -rf` line if you have local edits to keep.
#   - Local: imports from `../src` (notebook is in `notebooks/`).
import os, sys, subprocess

GITHUB_URL = 'https://github.com/meghna810/GPUBid.git'

if 'google.colab' in sys.modules:
    !rm -rf /content/GPUBid 2>/dev/null
    !git clone -q $GITHUB_URL /content/GPUBid
    !pip install --quiet pydantic numpy pulp plotly matplotlib ipywidgets anthropic openai tenacity
    REPO_ROOT = '/content/GPUBid'
    from google.colab import output
    output.enable_custom_widget_manager()
else:
    here = os.path.abspath('.')
    parent = os.path.abspath('..')
    candidates = [parent, here, '/content/GPUBid', '/content/drive/MyDrive/GPUBid']
    REPO_ROOT = next((c for c in candidates if os.path.isdir(os.path.join(c, 'src', 'gpubid'))), None)
    if REPO_ROOT is None:
        raise RuntimeError(
            "Couldn\'t find the gpubid package. Make sure you\'re running from the repo root, "
            "or run this notebook in Colab where it auto-clones from GitHub."
        )

# Evict any previously-imported gpubid modules so we don\'t pick up stale classes
# from an earlier run. Without this, re-running the setup cell pulls fresh files
# to disk but Python keeps the OLD classes in memory and you get cryptic
# AttributeErrors like "RoundSnapshot has no attribute \'actions\'".
# (For the deepest reset — including any variables that hold old class instances —
# use Runtime -> Restart runtime in Colab.)
for cached_name in [n for n in list(sys.modules) if n == 'gpubid' or n.startswith('gpubid.')]:
    del sys.modules[cached_name]

src_path = os.path.join(REPO_ROOT, 'src')
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)

import gpubid

def _git(args):
    try:
        return subprocess.run(['git', '-C', REPO_ROOT] + args,
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return None

commit_hash = _git(['rev-parse', '--short', 'HEAD']) or '?'
commit_subj = _git(['log', '-1', '--format=%s']) or '(no git info)'
print(f'gpubid v{gpubid.__version__}  ·  commit {commit_hash}: {commit_subj}')
print(f'  repo: {REPO_ROOT}')

## Mode + (optional) API key

Three ways to run a negotiation:

- **Fast** — deterministic rule-based agents. No API key. Instant. Good for exploring the sliders.
- **Preset** — replay a pre-baked LLM negotiation from `data/presets/`. No API key. Used in the video walkthrough.
- **Live** — real LLM agents. Paste your **Anthropic** (`sk-ant-…`) or **OpenAI** (`sk-…`) key below; provider is auto-detected. Costs a few cents per run. Your key stays in this notebook session.

Same code path drives all three — only the agent backend swaps.

In [ ]:
import ipywidgets as w
from gpubid.experiments.bake_presets import list_presets
from pathlib import Path

# Try to load an API key from Colab Secrets (the recommended way to provide it
# in Colab). Set ANTHROPIC_API_KEY or OPENAI_API_KEY in the Colab Secrets panel
# (key icon in the left sidebar). The Password field below is a fallback for
# local notebooks or one-off runs.
api_key_default = ''
if 'google.colab' in sys.modules:
    try:
        from google.colab import userdata
        for secret_name in ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY']:
            try:
                value = userdata.get(secret_name)
                if value:
                    api_key_default = value
                    print(f'✓ Loaded {secret_name} from Colab Secrets.')
                    break
            except Exception:
                continue  # secret not set; try the next
    except ImportError:
        pass

mode_select = w.Dropdown(
    options=['fast (deterministic — no key)', 'preset (LLM playback — no key)', 'live (real LLM — paste key)'],
    value='fast (deterministic — no key)',
    description='Mode',
    style={'description_width': 'initial'},
)
api_key_input = w.Password(
    value=api_key_default,
    placeholder='sk-ant-… or sk-… (or set in Colab Secrets)',
    description='API key',
    style={'description_width': 'initial'},
)
preset_paths = list_presets(Path(REPO_ROOT) / 'data' / 'presets')
preset_select = w.Dropdown(
    options=[(p.stem, str(p)) for p in preset_paths] or [('(no presets baked yet)', '')],
    description='Preset',
    style={'description_width': 'initial'},
)
w.VBox([mode_select, api_key_input, preset_select])

## 1. Pick a market

Move the sliders to choose how many buyers and sellers, the supply regime (`tight` = constrained, `slack` = generous), and a random seed.

_Note: in **preset** mode these sliders are ignored — the market comes from the preset file._

In [ ]:
buyers_slider  = w.IntSlider(value=8, min=3, max=12, description='# Buyers')
sellers_slider = w.IntSlider(value=4, min=2, max=6,  description='# Sellers')
regime_select  = w.Dropdown(options=['tight', 'slack'], value='tight', description='Regime')
seed_slider    = w.IntSlider(value=42, min=0, max=999, description='Seed')
rounds_slider  = w.IntSlider(value=5, min=1, max=10,  description='Max rounds')
w.VBox([buyers_slider, sellers_slider, regime_select, seed_slider, rounds_slider])

## 2. Render the market

Each buyer card shows the job's GPU types, qty, duration, time window, interruption tolerance, and an urgency bar (green = patient, red = urgent). Each seller card shows their capacity slots, with off-peak slots tagged. **Private** values are visible to *us* in the notebook for inspection — agents only see structured public offers.

In [ ]:
from gpubid.market import generate_market

market = generate_market(
    n_buyers=buyers_slider.value,
    n_sellers=sellers_slider.value,
    regime=regime_select.value,
    seed=seed_slider.value,
)
market

## 3. Watch the agents negotiate

Hit run. Sellers post asks, buyers post bids, and acceptances form deals — animated round-by-round. The deals pane at the bottom flashes green for new agreements.

In [ ]:
from gpubid.viz.trading_floor import animate_negotiation

mode_label = mode_select.value
if mode_label.startswith('fast'):
    final, market, all_snapshots = animate_negotiation(
        market, mode='fast',
        max_rounds=rounds_slider.value, step_seconds=1.0,
    )
elif mode_label.startswith('preset'):
    if not preset_select.value:
        raise RuntimeError(
            'No presets in data/presets/. Bake one with the team API keys: \n'
            '    ANTHROPIC_API_KEY=sk-ant-... PYTHONPATH=src python -m gpubid.experiments.bake_presets all'
        )
    final, market, all_snapshots = animate_negotiation(
        mode='preset', preset_path=preset_select.value, step_seconds=1.2,
    )
elif mode_label.startswith('live'):
    if not api_key_input.value:
        raise RuntimeError('Live mode requires an API key in the settings cell above.')
    final, market, all_snapshots = animate_negotiation(
        market, mode='live',
        api_key=api_key_input.value,
        max_rounds=rounds_slider.value, step_seconds=0.4,
    )
else:
    raise RuntimeError(f'Unknown mode: {mode_label}')

n_deals = len(final.all_deals)
total_surplus = sum(d.total_value for d in final.all_deals)
print(f'\n{n_deals} deals struck across {final.round_n} rounds, total transacted: ${total_surplus:.0f}')

## 4. Compare to baselines

How well did the agents do compared to the **welfare-optimal** allocation (VCG, computed by a mixed-integer program — what an oracle planner would assign) and to the **posted-price** baseline (a single take-it-or-leave-it price per GPU type — closer to what current cloud markets give you)?

In [ ]:
from gpubid.benchmark.vcg import solve_vcg
from gpubid.benchmark.posted_price import solve_posted_price
from gpubid.eval.metrics import compute_metrics
from gpubid.viz.charts import baseline_comparison, metric_table
from IPython.display import HTML, display

vcg_result    = solve_vcg(market)
posted_result = solve_posted_price(market)

agentic_metrics = compute_metrics(market, list(final.all_deals))
vcg_metrics     = compute_metrics(market, vcg_result.deals)
posted_metrics  = compute_metrics(market, posted_result.deals)

fig = baseline_comparison(
    agentic=agentic_metrics, vcg=vcg_metrics, posted=posted_metrics,
    title=f'Welfare comparison — {market.regime} supply (seed {market.seed})',
)
fig.show()
display(HTML(metric_table(agentic=agentic_metrics, vcg=vcg_metrics, posted=posted_metrics)))

if vcg_metrics.welfare > 0:
    recov  = agentic_metrics.welfare / vcg_metrics.welfare * 100
    pposted = posted_metrics.welfare / vcg_metrics.welfare * 100
    print(f'\nAgentic recovered {recov:.0f}% of VCG-optimal welfare; posted-price recovered {pposted:.0f}%.')

## 5. Tradeoff sandbox

Two toggles, live: the **concentration cap** and the **round count**. Drag the sliders and watch the welfare-recovery, Gini, and buyer-fill-rate update in real time. Same market as above (cell 2's seed). All in fast mode so it's instantaneous.

_The other two tradeoffs in the proposal — homogeneous-vs-heterogeneous sellers and structured-vs-free-form offers — only show their effect with real LLMs, so they appear in the headline figures (cell 7) and in dedicated presets._

In [ ]:
from gpubid.viz.trading_floor import collect_snapshots

vcg_for_sandbox = solve_vcg(market)
posted_for_sandbox = solve_posted_price(market)
posted_metrics_sb = compute_metrics(market, posted_for_sandbox.deals)

@w.interact(
    cap_pct=w.FloatSlider(value=0.0, min=0.0, max=0.25, step=0.025, description='Cap %', readout_format='.1%'),
    max_rounds=w.IntSlider(value=5, min=1, max=10, description='Max rounds'),
)
def explore_tradeoffs(cap_pct, max_rounds):
    cap = cap_pct if cap_pct > 0 else None
    snaps = collect_snapshots(market, max_rounds=max_rounds, concentration_cap_pct=cap)
    deals = list(snaps[-1].all_deals)
    m = compute_metrics(market, deals)
    recov = m.welfare / vcg_for_sandbox.welfare * 100 if vcg_for_sandbox.welfare > 0 else 0
    posted_recov = posted_metrics_sb.welfare / vcg_for_sandbox.welfare * 100 if vcg_for_sandbox.welfare > 0 else 0

    def stat(label, value, hint=''):
        return (f'<div style="flex:1;text-align:center;background:#fff;border:1px solid #e5e7eb;'
                f'border-radius:8px;padding:10px;">'
                f'<div style="font-size:11px;color:#6b7280;text-transform:uppercase;">{label}</div>'
                f'<div style="font-size:22px;font-weight:600;color:#111;">{value}</div>'
                f'<div style="font-size:10px;color:#9ca3af;">{hint}</div>'
                f'</div>')

    html = (f'<div style="display:flex;gap:8px;font-family:-apple-system, sans-serif;margin:8px 0;">'
            f'{stat("Recovery", f"{recov:.0f}%", f"vs posted {posted_recov:.0f}%")}'
            f'{stat("Buyers filled", f"{m.n_buyers_filled}/{len(market.buyers)}")}'
            f'{stat("Avg price", f"${m.avg_clearing_price:.2f}")}'
            f'{stat("Gini", f"{m.gini_buyer_welfare:.3f}", "buyer welfare inequality")}'
            f'{stat("Off-peak util", f"{m.offpeak_utilization*100:.0f}%")}'
            f'</div>')
    display(HTML(html))

## 6. Inspect a deal — bilateral negotiation thread

Pick any deal from the run above. Below the deal summary, you'll see the *implicit conversation* between the buyer and the seller's slot — only the offers from those two parties, in round order, like a 1:1 chat thread.

The full marketplace tape (everyone shouting their offers) is in cell 6.6. **This view is the bilateral subset** — the negotiation that actually concluded the deal.

In live LLM mode the reasoning text on each bubble is genuine model output, so the conversation reads more like an actual back-and-forth ("I'm matching your $3.40 ask because the round-4 deadline is approaching" etc.). In fast mode it's deterministic stub text.

In [ ]:
from gpubid.viz.trace_view import render_trace

if not final.all_deals:
    display(HTML('<em>No deals to inspect — try a slack-supply market or more rounds.</em>'))
else:
    deal_options = [(f'{d.buyer_id} ↔ {d.seller_id} · {d.gpu_type.value}×{d.qty} · ${d.price_per_gpu_hr:.2f}/hr (round {d.round_n})', d.id) for d in final.all_deals]
    deal_select = w.Dropdown(options=deal_options, description='Deal')

    out = w.Output()
    def _on_change(change):
        deal = next(d for d in final.all_deals if d.id == change['new'])
        out.clear_output(wait=True)
        with out:
            display(HTML(render_trace(deal, market, snapshots=all_snapshots)))
    deal_select.observe(_on_change, names='value')
    # Initial render — passes all_snapshots so the bilateral negotiation thread
    # (just this buyer ↔ this seller's slot, in chronological order) renders.
    with out:
        display(HTML(render_trace(final.all_deals[0], market, snapshots=all_snapshots)))
    display(deal_select, out)

## 6.5 Negotiation forensics — who held out, who conceded?

The animation in cell 3 shows offers in flight; this view reconstructs the round-by-round actions every agent took, with two diagnostics:

- **Timeline chart**: each agent's posted price across rounds. Solid line = won a deal; dashed = held out. Stars mark where deals cleared.
- **Aggression scoreboard**: how much each agent moved from their first offer to their last. Buyers in `+%` climbed (raised bids); sellers in `+%` decayed (dropped asks).

Below those: a per-round log of every action with ↑/↓ arrows showing direction.

In [ ]:
from gpubid.analysis.forensics import (
    extract_history, render_timeline, render_aggression_scoreboard, render_log,
)

# Reuse `all_snapshots` from cell 3 (the animation cell). In live mode this avoids
# re-paying for the LLM calls; in fast mode it's just faster.
history = extract_history(market, all_snapshots)

# 1) Offer trajectories
render_timeline(history).show()

# 2) Aggression scoreboard
display(HTML(render_aggression_scoreboard(history)))

# 3) Round-by-round log (collapsed by default, scroll to read)
display(HTML(
    '<details><summary style="cursor:pointer;font-size:13px;color:#374151;font-weight:600;">'
    'Round-by-round log (click to expand)</summary>' + render_log(history) + '</details>'
))

## 6.6 Chat exchange — what each agent actually said

The most interesting part of the negotiation: **the reasoning text the agents wrote**. In live LLM mode this is genuine model output — the agent's stated strategy, hedging, second-guessing. In fast mode the deterministic agents just stamp something like `"opening ask at $5.40"`, but the same renderer works for both.

Bubbles aligned right are buyers; left are sellers. Color-coded by side. Toggle `only_with_reasoning=True` if you want to skip the deterministic stubs and only see LLM-written messages.

In [ ]:
from gpubid.analysis.forensics import render_chat_exchange

# `history` was extracted in cell 6.5 above.
display(HTML(render_chat_exchange(history, only_with_reasoning=False, max_height_px=600)))

## 6.7 Provider tournament — Claude vs OpenAI head-to-head

Run the same market under different LLM-provider assignments to answer:

- **Who wins more deals — Claude buyers or OpenAI buyers?**
- **Are Claude sellers more aggressive at decaying prices than OpenAI sellers?**
- **Does the welfare-vs-collusion gap show up at this sample size?**

The default uses round-robin assignment: half the buyers go on Anthropic, half on OpenAI; same for sellers. Each seed is a different synthetic market, so we're averaging over market conditions, not just one shape.

⚠️ **This cell makes real LLM calls and costs API credits.** 5 seeds × 12 agents × 5 rounds ≈ 300 calls (~$1-2 with default cost-effective models). Default is a no-key smoke test (deterministic agents tagged with synthetic provider labels) so the framework is exercised without spending. **Switch `mode` to `"live"` and set `n_seeds` higher to run the real comparison.**

In [ ]:
import os
from gpubid.analysis.tournament import (
    head_to_head_deterministic, head_to_head_alternating,
    render_tournament_report, render_tournament_chart,
)

# Configurable knobs — bump n_seeds=10 once you've got both keys in Colab Secrets.
MODE = "deterministic"   # set to "live" for real LLM calls (requires both keys)
N_SEEDS = 5
N_BUYERS = 8
N_SELLERS = 4
REGIME = "tight"
MAX_ROUNDS = 5

if MODE == "live":
    api_keys = {}
    if 'google.colab' in sys.modules:
        try:
            from google.colab import userdata
            for prov, name in [("anthropic", "ANTHROPIC_API_KEY"), ("openai", "OPENAI_API_KEY")]:
                try:
                    v = userdata.get(name)
                    if v: api_keys[prov] = v
                except Exception:
                    pass
        except ImportError:
            pass
    if not api_keys.get("anthropic") or not api_keys.get("openai"):
        # Fall back to env vars (local) or to whatever was pasted into the password field.
        for prov, env in [("anthropic", "ANTHROPIC_API_KEY"), ("openai", "OPENAI_API_KEY")]:
            v = os.environ.get(env)
            if v: api_keys.setdefault(prov, v)
    if not api_keys.get("anthropic") or not api_keys.get("openai"):
        raise RuntimeError(
            "Live tournament needs BOTH Anthropic AND OpenAI keys. "
            "Set ANTHROPIC_API_KEY and OPENAI_API_KEY in Colab Secrets (or env vars)."
        )

    result = head_to_head_alternating(
        n_seeds=N_SEEDS, api_keys=api_keys,
        n_buyers=N_BUYERS, n_sellers=N_SELLERS, regime=REGIME, max_rounds=MAX_ROUNDS,
    )
else:
    result = head_to_head_deterministic(
        n_seeds=N_SEEDS, n_buyers=N_BUYERS, n_sellers=N_SELLERS, regime=REGIME, max_rounds=MAX_ROUNDS,
    )

display(HTML(render_tournament_report(result)))
render_tournament_chart(result).show()

## 7. Headline figures

Pre-rendered figures from `data/figures/`, regenerable via `PYTHONPATH=src python -m gpubid.experiments.run_sweep`.

These are the four figures that go on the one-pager:
1. **Welfare recovery vs round count** — diminishing returns; N=5 sits on the elbow.
2. **Welfare and Gini vs concentration cap** — fairness vs welfare tradeoff.
3. **Agentic vs posted-price** — across 15+ seeds in both regimes.
4. **Off-peak slot utilization** — agentic fills off-peak; posted-price doesn't.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

figures_dir = Path(REPO_ROOT) / 'data' / 'figures'
for name in ['welfare_vs_rounds.png', 'welfare_vs_cap.png', 'agentic_vs_posted.png', 'offpeak_utilization.png']:
    path = figures_dir / name
    if path.exists():
        display(Image(str(path)))
    else:
        display(HTML(f'<em>Missing {name} — regenerate with:'
                     f' <code>PYTHONPATH=src python -m gpubid.experiments.run_sweep</code></em>'))

## 8. Writeup

**Problem.** GPU cloud markets force buyers into three rigid contract types: long-term reserved, on-demand, or spot. A team that needs 8 H100s in the next hour pays the same on-demand rate as a team willing to wait 12 hours, even though they impose very different claims on perishable capacity. The middle of the demand curve is unserved.

**Mechanism.** Buyer agents and seller agents post structured offer tuples (price, qty, GPU type, time slot, duration, interruption tolerance) plus free-form reasoning. They negotiate over rounds via a public board. A central engine clears agreed deals, enforces a concentration cap, and validates against private reserves.

**Headline result.** Across 15+ seeds and both supply regimes, the agentic mechanism recovers around 90% of the VCG welfare upper bound, beating the posted-price baseline by a wide margin in slack-supply markets where off-peak slots otherwise go unfilled.

**Tradeoffs.** See the proposal: structure vs autonomy (we chose structured tuples + free-form reasoning), welfare vs collusion (heterogeneous backends — provider mix toggle in M5), expressiveness vs tractability (6-dim bids, default 8×4 markets), and fairness vs revenue (20% concentration cap). The first and second require LLM agents to demonstrate; baked presets in `data/presets/` cover them.